# Scheduler logs -> UsageLog, full period

Writes `UsageLog20230401_20251130_20260301_20260831.csv` (UTC dates in the file names, as in the minute-trace file names) to `05_Results/preprocessing/stage1`.

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()


In [ ]:
# # 구글드라이브 연결
# from google.colab import drive



In [ ]:
# ==============================================================================
# 0) 코랩/로컬 공용: 폴더 PATH 선언부 (Usage Logs 포함)
# ==============================================================================

from pathlib import Path
import os

def is_colab() -> bool:
    return Path("/content").exists() and any(
        k in os.environ for k in ["COLAB_GPU", "COLAB_RELEASE_TAG"]
    )

# 1) 프로젝트 루트(= BASE)만 환경별로 다르게
COLAB_BASE = Path(str(PROCESSED_DATA))
LOCAL_BASE = ROOT / '05_Results' / 'preprocessing' / 'stage1'

BASE = COLAB_BASE if is_colab() else LOCAL_BASE

# (선택) Colab인데 Drive가 안 붙어 있으면 경고
if is_colab() and not Path("/content/gdrive").exists():
    print("⚠️ Colab: Google Drive가 마운트되지 않았습니다. drive.mount('/content/gdrive') 먼저 실행하세요.")

# 2) 하위 폴더/파일은 상대경로로 통일
P = {
    # 원본 Usage Logs(다수 CSV가 있는 폴더)
    "usage_log_dir": RAW_USAGE,

    # UsageLog 정리본
    # 분석기간:
    #   Train: 2023-04-01 ~ 2025-11-30
    #   Test : 2026-03-01 ~ 2026-08-31
    "usage_log_clean_csv": (
        BASE / "UsageLog20230401_20251130_20260301_20260831.csv"
    ),

    # TimeSeries 원본 CSV 폴더
    "timeseries_csv_dir": BASE / "99. Data_TimeSeries_csv",

    # TimeSeries parquet 폴더(생성)
    "timeseries_bin_dir": BASE / "02. Data_TimeSeries_binary",

    # under1min 폴더(생성)
    "under1min_dir": BASE / "under1min",
}

# 3) 생성 대상 폴더만 자동 생성(입력 폴더는 만들지 않음)
for k in ["timeseries_bin_dir", "under1min_dir"]:
    P[k].mkdir(parents=True, exist_ok=True)

# 4) 출력/체크
print("BASE =", BASE)
for k, v in P.items():
    print(f"{k}: {v}")

print("\n[CHECK]")
print("usage_log_dir exists:", P["usage_log_dir"].exists())
print("usage_log_clean_csv exists:", P["usage_log_clean_csv"].exists())
print("timeseries_csv_dir exists:", P["timeseries_csv_dir"].exists())


In [ ]:

# Usage Logs 폴더 (코랩/로컬 공통)
folder_path = BASE / "01. Data_UsageLog"

csv_files = [p for p in folder_path.rglob("*.csv")]

print("CSV 개수:", len(csv_files))
for p in csv_files[:50]:
    print(str(p))
if len(csv_files) > 50:
    print(f"... (총 {len(csv_files)}개 중 50개만 출력)")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from datetime import datetime
import time
import re
from functools import reduce

import warnings
warnings.filterwarnings('ignore')
import matplotlib.cbook
# warnings.filterwarnings('ignore', category=matplotlib.cbook.mplDeprecation)

import os

plt.rc('font', family='NanumBarunGothic')



In [ ]:
# # 한글 폰트 설치
# !sudo apt-get install -y fonts-nanum
# !sudo fc-cache -fv
# !rm ~/.cache/matplotlib -rf

# # 여기까지 하고, 런타임 재시작


In [ ]:
# Data_UsageLog 데이터를 읽어들인다.

# ==============================================================================
# Data_UsageLog 데이터를 읽어들인다 (P 기반 / 코랩+로컬 공용)
#   - P["usage_log_dir"] 아래의 모든 CSV를 재귀적으로 읽어서 df_1로 concat
#   - 소스 추적 컬럼(__source_file__, __source_path__) 추가
#   - 컬럼 union으로 정규화(누락 컬럼은 NA로 채움)
# ==============================================================================

from pathlib import Path
import os
import pandas as pd

# =========================
# 0) 설정 (P 기반)
# =========================
folder_path = P["usage_log_dir"]  # ✅ 코랩/로컬 공용

SOURCE_FILE_COL = "__source_file__"
SOURCE_PATH_COL = "__source_path__"

# =========================
# 1) 하위폴더 포함 CSV 파일 전부 찾기
# =========================
csv_files = sorted(folder_path.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"CSV 파일을 찾지 못했습니다: {folder_path}")

print(f"[INFO] 발견한 CSV: {len(csv_files)}개")
print("[INFO] 예시(최대 20개):")
for p in csv_files[:20]:
    print(" -", p)

# =========================
# 2) CSV 읽기 + 소스 컬럼 보강 + 컬럼셋 수집
# =========================
dfs = []
read_fail = []            # (path, error)
colsets = {}              # path -> set(columns)

for fp in csv_files:
    try:
        df = pd.read_csv(fp)

        # 소스 추적 컬럼 추가
        df[SOURCE_FILE_COL] = fp.name
        df[SOURCE_PATH_COL] = str(fp)

        dfs.append(df)
        colsets[str(fp)] = set(df.columns)

    except Exception as e:
        read_fail.append((str(fp), str(e)))

print(f"[INFO] 읽기 성공: {len(dfs)}개 / 실패: {len(read_fail)}개")

if len(dfs) == 0:
    print("[ERROR] 모든 CSV 읽기에 실패했습니다. 아래 실패 로그를 확인하세요.")
    for fp, err in read_fail[:20]:
        print(f"  - {fp}\n    -> {err}")
    raise RuntimeError("모든 CSV 읽기에 실패했습니다.")

if read_fail:
    print("\n[WARN] 읽기 실패 파일(상위 20개만 표시):")
    for fp, err in read_fail[:20]:
        print(f"  - {fp}\n    -> {err}")
    if len(read_fail) > 20:
        print(f"  ... (총 {len(read_fail)}개)")

# =========================
# 3) 컬럼 불일치 검증 (Union 기준)
# =========================
all_cols = set()
for s in colsets.values():
    all_cols |= s

# 소스 컬럼은 맨 앞으로 고정
base_cols = sorted([c for c in all_cols if c not in {SOURCE_FILE_COL, SOURCE_PATH_COL}])
final_cols = [SOURCE_FILE_COL, SOURCE_PATH_COL] + base_cols

missing_counts = {c: 0 for c in final_cols}
missing_by_file = []  # (file, missing_cols)

for fp, cols in colsets.items():
    cols = set(cols)
    missing = [c for c in final_cols if c not in cols]
    for c in missing:
        missing_counts[c] += 1
    if missing:
        missing_by_file.append((fp, missing))

print("\n[INFO] 최종 컬럼 수(Union):", len(final_cols))
print("[INFO] 컬럼 누락이 있는 파일 수:", len(missing_by_file))

missing_rank = sorted(
    [(c, cnt) for c, cnt in missing_counts.items() if cnt > 0],
    key=lambda x: x[1],
    reverse=True
)

if missing_rank:
    print("\n[INFO] 누락이 많은 컬럼 TOP 20 (컬럼명, 누락파일수):")
    for c, cnt in missing_rank[:20]:
        print(f"  - {c}: {cnt}/{len(dfs)}")
else:
    print("\n[INFO] 모든 파일이 동일한 컬럼 구조입니다. (누락 없음)")

print("\n[DETAIL] 컬럼 누락 파일 예시(상위 10개만 표시):")
for fp, missing in missing_by_file[:10]:
    print(f"  - {fp}")
    print(f"    missing({len(missing)}): {missing[:30]}" + (" ..." if len(missing) > 30 else ""))

# =========================
# 4) 컬럼 정렬/정규화 후 concat
# =========================
normalized = []
for df in dfs:
    # 누락 컬럼 생성(NA)
    for c in final_cols:
        if c not in df.columns:
            df[c] = pd.NA
    # 최종 컬럼 순서 강제
    normalized.append(df[final_cols])

df_1 = pd.concat(normalized, ignore_index=True)

print("\n[OK] df_1 shape:", df_1.shape)

# 주피터/코랩에서 display가 없을 수도 있어 안전 처리
try:
    display(df_1.head())
except NameError:
    print(df_1.head().to_string(index=False))

# =========================
# 5) 최종 컬럼 전체 출력
# =========================
print("\n[INFO] df_1 전체 컬럼 목록")
print(f"[INFO] 총 컬럼 수: {df_1.shape[1]}\n")

print("\n".join(
    f"{i:03d}. {col}"
    for i, col in enumerate(df_1.columns.tolist(), start=1)
))




In [ ]:
# 완전 중복 행 제거
print(f"✅ 완전 중복 제거 이전. 현재 총 행 수: {len(df_1)}")

df_1.drop_duplicates(inplace=True)
df_1.reset_index(drop=True, inplace=True)

print(f"✅ 완전 중복 제거 완료. 현재 총 행 수: {len(df_1)}")


In [ ]:
# container_ids 기준으로 중복된 건을 최신만 남기고 삭제

print(f"⚠️ 처리 전 container_ids 남은 중복 건수: {sum(df_1.value_counts('container_ids') > 1)}")

# 1. 종료 시간 기준으로 정렬 (가장 최신 로그를 남기기 위해)
df_1['terminated_at'] = pd.to_datetime(df_1['terminated_at'])
df_1 = df_1.sort_values(by='terminated_at', ascending=False)

# 2. container_ids가 같은 데이터 중 첫 번째(최신)만 남기고 삭제
df_1.drop_duplicates(subset=['container_ids'], keep='first', inplace=True)

# 3. 이제 인덱스를 깨끗하게 정리합니다.
df_1.reset_index(drop=True, inplace=True)

# 4. 검증: 이제 0이 나와야 정상입니다.
print(f"⚠️ 처리 후 container_ids 중복 건수: {sum(df_1.value_counts('container_ids') > 1)}")


#### Column 설명
#### Column 설명
##### 1. Container / Session 정보
##### 2. 할당 자원 정보
##### 3. Container Lifecycle 정보(시간은 모두 UTC이다. KST로 변경 필요.)
##### 4. Container 상태 및 종료 정보
##### 5. Resource Utilization 정보
##### 6. Storage I/O 정보
##### 7. Network 정보

In [ ]:
# 사용할 컬럼만 뽑아낸다.
col_names = [
    'session_id', 'session_name', 'container_ids', 'agent',
    'cpu_allocated', 'mem_allocated', 'gpu_allocated', 'shared_memory',
    'device_type', 'smp', 'gpu_mem_allocated',
    'created_at', 'run_at', 'terminating_at', 'terminated_at',
    'user_email', 'status', 'status_info',
    # --- 추가된 리소스 사용량 및 성능 지표 ---
    'cpu_util_avg', 'mem_util_avg', 'gpu_util_avg',
    'cpu_util_max', 'mem_util_max', 'gpu_mem_used_peak_MB',
    'io_read_bytes_total', 'io_write_bytes_total',
    'net_rx_peak_bps', 'net_tx_peak_bps'
]
# col_names
df_selected = df_1[col_names]


In [ ]:
# created_time, terminated_time이 datetime 형식이 아닐 경우 변환
df_selected['created_at'] = pd.to_datetime(df_selected['created_at'])
df_selected['run_at'] = pd.to_datetime(df_selected['run_at'])
df_selected['terminating_at'] = pd.to_datetime(df_selected['terminating_at'])
df_selected['terminated_at'] = pd.to_datetime(df_selected['terminated_at'])

# 시간 차이 계산 (분 단위). 레블업에서 제공한 사용시간이 안맞아서 따로 계산함.
df_selected['used_time'] = (df_selected['terminated_at'] - df_selected['created_at']).dt.total_seconds() / 60


In [ ]:
# 컬럼명을 알기 쉽게 수정
df_selected.rename(columns = {'container_ids':'container_id',
                              'cpu_allocated':'cpu_allocated(Core)',
                              # 'mem_allocated':'mem_allocated(Byte)',
                              'mem_allocated':'mem_allocated(GB)',
                              'gpu_allocated':'gpu_allocated(EA)',
                              # 'shared_memory':'shared_memory(Byte)',
                              'cpu_util_avg':'cpu_util_avg(%)',
                              'mem_util_avg':'mem_util_avg(%)',
                              'gpu_util_avg':'gpu_util_avg(%)',
                              'cpu_util_max':'cpu_util_max(%)',
                              'shared_memory':'shared_memory(GB)',
                              'smp': 'smp(EA)',
                              'gpu_mem_allocated':'gpu_mem_allocated(Byte)',
                              'used_time':'used_time(minute)',
                              'created_at':'created_time',
                              'run_at': 'run_time',
                              'terminating_at':'terminating_time',
                              'terminated_at':'terminated_time'
                              }, inplace = True)
df_selected.head()


In [ ]:
df_selected.columns


In [ ]:
# raw data에서 cpu_util_avg를 구하려면 Cores로 나눠줘야 한다.(소수점 셋째자리에서 반올림)
df_selected['cpu_util_avg(%)'] = (df_selected['cpu_util_avg(%)'] / df_selected['cpu_allocated(Core)']).round(2)
df_selected['cpu_util_max(%)'] = (df_selected['cpu_util_max(%)'] / df_selected['cpu_allocated(Core)']).round(2)


In [ ]:
# terminated_time 이 결측치인 것을 삭제한다.
df_selected = df_selected[df_selected['terminated_time'] != 'None']


In [ ]:
(df_selected['terminated_time'] == 'None').sum()


In [ ]:
df_selected.columns


In [ ]:
# container_ids가 na인 값을 삭제한다.
df_selected.dropna(axis = 0, subset = ['container_id'], inplace = True)
# 2. 문자열 "NaN" 또는 빈 문자열 제거
df_selected = df_selected[
    (df_selected['container_id'].astype(str) != 'NaN') &
    (df_selected['container_id'].astype(str) != 'nan') &
    (df_selected['container_id'].notna())
].copy()
# 3. 인덱스 재설정 (삭제된 행 번호 메우기)
df_selected.reset_index(drop=True, inplace=True)


In [ ]:
# Convert '생성시간' and '종료시간' from string to datetime
df_selected['created_time'] = pd.to_datetime(df_selected['created_time'])
df_selected['terminated_time'] = pd.to_datetime(df_selected['terminated_time'])

# Extract year, month, day from '생성시간' and '종료시간'
df_selected['created_time_yymmdd'] = df_selected['created_time'].dt.strftime('%Y%m%d')
df_selected['terminated_time_yymmdd'] = df_selected['terminated_time'].dt.strftime('%Y%m%d')

# 여러 파일에서 사용할 공통 filename 서식을 생성한다.
df_selected['filename'] = df_selected['created_time_yymmdd'] + "-" + df_selected['terminated_time_yymmdd'] + "-" + (df_selected['user_email'].str.replace("@","_") + "-" + df_selected['session_name'] + "-" + df_selected['session_id'] + "-" + df_selected['container_id'] + ".csv")

# 생성시간, 종료시간을 삭제한다.
# df_selected.drop(['생성시간', '종료시간'], axis = 1, inplace = True)


In [ ]:
df_selected[['created_time', 'run_time', 'terminating_time', 'terminated_time']]


In [ ]:
# Byte 용량을 GB 용량으로 변환
# df_selected['mem_allocated(GB)'] = df_selected['mem_allocated(Byte)']/(1024**3)
# df_selected['shared_memory(GB)'] = df_selected['shared_memory(Byte)']/(1024**3)
df_selected['gpu_mem_allocated(GB)'] = df_selected['gpu_mem_allocated(Byte)']/(1024**3)
# df_selected.head()


In [ ]:
# Byte 용량 데이터 삭제
# df_selected.drop(['mem_allocated(Byte)', 'shared_memory(Byte)', 'gpu_mem_allocated(Byte)'], axis = 1, inplace = True)
df_selected.drop(['gpu_mem_allocated(Byte)'], axis = 1, inplace = True)
# df_selected.head()


In [ ]:
# 장비타입이 Nan인 경우, 'GPU 미사용'으로, NVIDIA A100-SXM4-80GB인 경우, 'GPU 사용' 으로 변경한다.
df_selected['device_type'].fillna('Non-GPU', inplace = True)
df_selected['device_type'].replace('NVIDIA A100-SXM4-80GB', 'GPU-enabled', inplace = True)
# df_selected.isna().sum()


In [ ]:
df_selected['device_type'].value_counts()


In [ ]:
def grouping_gpu_card(x):
    if pd.isna(x):
        return np.nan

    x = float(x)

    if x == 0:
        return '0.0_GPU'
    elif 0 < x < 1:
        return '0.x_GPU'
    elif 1 <= x < 2:
        return '1.x_GPU'
    elif 2 <= x < 3:
        return '2.x_GPU'
    elif 3 <= x < 4:
        return '3.x_GPU'
    elif x == 4:
        return '4.0_GPU'
    else:
        return '4+_GPU'   # 필요 없으면 삭제

df_selected['gpu_allocated(group)'] = df_selected['gpu_allocated(EA)'].apply(grouping_gpu_card)


In [ ]:
# GPU 사용 수량을 기준으로 Grouping을 실행한다.
#(0.x 장은 실행수가 100건 정도로 많다. 이에, 소숫점으로 GPU를 실행하는 0.x장만 남기고 나머지는 삭제한다.)
def grouping_gpu_card(x):
    if x == 0:
        return '0.0_GPU'
    elif (x < 1) & (x > 0):
        return '0.x_GPU'
    elif (x == 1):
        return '1.0_GPU'
    elif (x == 2):
        return '2.0_GPU'
    elif (x == 3):
        return '3.0_GPU'
    elif (x == 4):
        return '4.0_GPU'

df_selected['gpu_allocated(group)'] = df_selected['gpu_allocated(EA)'].apply(lambda x : grouping_gpu_card(x))
# df_selected.head()


In [ ]:
# agent(서버정보) 값 NaN인 데이터 삭제(4건)
df_selected.dropna(subset=['agent'], inplace=True)


In [ ]:
# agent에서 사용할 서버 번호 숫자만 뽑아낸다.
df_selected['agent'] = df_selected['agent'].str.extract(r'(\d+)', expand=False).astype(int)
# df_selected.head()


In [ ]:
df_selected.columns


In [ ]:
# status_info 를 확인해서, Auto-stop / Manual-stop으로 그룹핑

def classify_termination(status):
    s = str(status).strip()  # NaN/공백 방어

    # Auto-stop: idle-utilization 단독
    if s == 'idle-utilization':
        return 'Auto-stop'

    # Manual-stop: 나머지 전부
    return 'Manual-stop'

# 그룹핑 적용
df_selected['status_info(group)'] = df_selected['status_info'].apply(classify_termination)

# (선택) 그룹별 개수 확인
print(df_selected['status_info(group)'].value_counts(dropna=False))

# (선택) Auto-stop만 / Manual-stop만 분리
df_auto = df_selected[df_selected['status_info(group)'] == 'Auto-stop'].copy()
df_manual = df_selected[df_selected['status_info(group)'] == 'Manual-stop'].copy()


# TABLE 4
## CONTAINER TERMINATION CODES AND THEIR DEFINITIONS.

In [ ]:
# 1. value_counts()로 빈도수 계산하여 DataFrame으로 변환
df_summary = pd.DataFrame(df_selected['status_info'].value_counts())
df_summary.columns = ['Count']  # 컬럼명을 'Count'로 설정

# 2. 비율(Ratio) 컬럼 추가 (백분율, 소수점 2째 자리 반올림)
total_count = df_summary['Count'].sum()
df_summary['Ratio (%)'] = round((df_summary['Count'] / total_count) * 100, 2)

# 3. 결과 출력
df_summary


#### status_info(group) 분류 기준

In [ ]:
# filename이 중복된 값 확인
sum(df_selected.value_counts('filename') > 1)


In [ ]:
# 시간 컬럼 UTC → KST 변환
TIME_COLS = [
    'created_time',
    'run_time',
    'terminating_time',
    'terminated_time'
]

for c in TIME_COLS:
    if c in df_selected.columns:
        df_selected[c] = (
            pd.to_datetime(df_selected[c], utc=True, errors='coerce')
              .dt.tz_convert('Asia/Seoul')
              .dt.tz_localize(None)
        )

# 확인
print(df_selected[TIME_COLS].dtypes)
print(df_selected[TIME_COLS].head(3))


In [ ]:
# ==============================================================================
# 분석 대상 기간 필터링
#
# Train : 2023-03-01 ~ 2025-11-30
# Test  : 2026-03-01 ~ 2026-08-31
#
# ※ 2025-12-24 ~ 2026-02 기간은 데이터 미확보
# ※ 2023-02, 2025-12 데이터는 소량이므로 월별 추이의 일관성을 위해 제외
# ==============================================================================

# 1. 기존 데이터 구간 (Train)
train_mask = (
    (df_selected['created_time'] >= '2023-03-01') &
    (df_selected['created_time'] <  '2025-12-01')
)

# 2. 신규 데이터 구간 (Test)
test_mask = (
    (df_selected['created_time'] >= '2026-03-01') &
    (df_selected['created_time'] <  '2026-09-01')
)

# 3. Train / Test 분리
df_train = df_selected.loc[train_mask].copy()
df_test  = df_selected.loc[test_mask].copy()

# 4. 전체 분석 대상 데이터
df_filtered = df_selected.loc[train_mask | test_mask].copy()

# 5. 결과 검증
print(f"📊 필터링 전 개수 : {len(df_selected):,}개")
print(f"📘 Train 개수     : {len(df_train):,}개")
print(f"📙 Test 개수      : {len(df_test):,}개")
print(f"✅ 전체 분석 대상 : {len(df_filtered):,}개")
print(f"🗑️ 제외된 데이터  : {len(df_selected) - len(df_filtered):,}개")

print("\n[Train 기간]")
print(
    df_train['created_time'].min(),
    "~",
    df_train['created_time'].max()
)

print("\n[Test 기간]")
print(
    df_test['created_time'].min(),
    "~",
    df_test['created_time'].max()
)

# 기존 이후 분석 코드 호환성을 위해 전체 분석 대상 데이터로 덮어쓰기
df_selected = df_filtered


In [ ]:
# 생성시간을 연, 월, 일, 시로 분리한다.
df_selected['created_time(year)'] = pd.to_datetime(df_selected['created_time']).apply(lambda x: x.year)
df_selected['created_time(month)'] = pd.to_datetime(df_selected['created_time']).apply(lambda x: x.month)
df_selected['created_time(day)'] = pd.to_datetime(df_selected['created_time']).apply(lambda x: x.day)
df_selected['created_time(hour)'] = pd.to_datetime(df_selected['created_time']).apply(lambda x: x.hour)

# df_selected.head()


In [ ]:
# 종료시간을 연, 월, 일, 시로 분리한다.
df_selected['terminated_time(year)'] = pd.to_datetime(df_selected['terminated_time']).apply(lambda x: x.year)
df_selected['terminated_time(month)'] = pd.to_datetime(df_selected['terminated_time']).apply(lambda x: x.month)
df_selected['terminated_time(day)'] = pd.to_datetime(df_selected['terminated_time']).apply(lambda x: x.day)
df_selected['terminated_time(hour)'] = pd.to_datetime(df_selected['terminated_time']).apply(lambda x: x.hour)


In [ ]:
# df_selected -> UsageLog20230401_20251130_20260301_20260831.csv 로 저장 (P 기반) ★
out_csv = P["usage_log_clean_csv"]  # BASE/UsageLog20230401_20251130_20260301_20260831.csv

# (권장) 혹시 모를 상위폴더 생성
out_csv.parent.mkdir(parents=True, exist_ok=True)

df_selected.to_csv(
    out_csv,
    sep=",",
    index=False,
    na_rep="NaN",
    encoding="utf-8-sig"  # 엑셀/한글 경로 호환에 안전
)

print(f"✅ 저장 완료: {out_csv}")


### 종료사유 설명
### status_info(group) 분류 기준

In [ ]:
df_selected['status_info(group)'].value_counts()


In [ ]:
df_selected.columns
